# 現代卷積神經網路架構

在學習了CNN的基礎概念和LeNet之後，讓我們探索推動深度學習革命的現代CNN架構。

## 本章內容

1. **AlexNet (2012)** - 開啟深度學習時代
2. **VGG (2014)** - 更深的網路
3. **Network in Network (NiN)** - 1×1卷積的應用
4. **GoogLeNet/Inception (2014)** - 多尺度特徵提取
5. **ResNet (2015)** - 殘差連接突破深度限制
6. **DenseNet** - 密集連接網路

## 學習目標

- 理解每個架構的創新點和設計思想
- 掌握如何實現這些架構
- 了解各架構的適用場景和性能特點
- 學會使用PyTorch構建和訓練這些模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import time

# 設置隨機種子以確保可重現性
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# 檢測可用的設備
device = torch.device('cuda' if torch.cuda.is_available() else 
                     'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'使用設備: {device}')

## 1. AlexNet (2012)

### 背景

AlexNet在2012年ImageNet圖像識別挑戰賽(ILSVRC)中以顯著優勢獲勝，top-5錯誤率從26%降至15.3%，開啟了深度學習時代。

### 創新點

1. **更深的網路**：8層（5個卷積層 + 3個全連接層）
2. **ReLU激活函數**：替代傳統的Sigmoid/Tanh，加速訓練
3. **Dropout正則化**：減少過擬合
4. **數據增強**：翻轉、裁剪、顏色抖動
5. **GPU並行計算**：使用兩塊GPU訓練
6. **局部響應歸一化(LRN)**：增強模型的泛化能力

### 網路結構

```
輸入: 224×224×3
├─ Conv1: 11×11, stride=4, pad=2 → 96 filters → 55×55×96
├─ MaxPool: 3×3, stride=2 → 27×27×96
├─ Conv2: 5×5, pad=2 → 256 filters → 27×27×256
├─ MaxPool: 3×3, stride=2 → 13×13×256
├─ Conv3: 3×3, pad=1 → 384 filters → 13×13×384
├─ Conv4: 3×3, pad=1 → 384 filters → 13×13×384
├─ Conv5: 3×3, pad=1 → 256 filters → 13×13×256
├─ MaxPool: 3×3, stride=2 → 6×6×256
├─ FC1: 4096
├─ FC2: 4096
└─ FC3: 1000 (ImageNet類別數)
```

In [ ]:
class AlexNet(nn.Module):
    """AlexNet實現（適配較小的輸入圖像）"""
    
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        
        # 特徵提取層
        self.features = nn.Sequential(
            # 第一層卷積
            nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # 第二層卷積
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # 第三層卷積
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # 第四層卷積
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # 第五層卷積
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        
        # 自適應池化層，確保輸出固定大小
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        
        # 分類器
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            
            nn.Linear(4096, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# 創建模型並查看結構
alexnet = AlexNet(num_classes=10)
print("AlexNet架構:")
print(alexnet)

# 測試前向傳播
x = torch.randn(1, 1, 224, 224)
output = alexnet(x)
print(f"\n輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")

# 計算參數量
total_params = sum(p.numel() for p in alexnet.parameters())
trainable_params = sum(p.numel() for p in alexnet.parameters() if p.requires_grad)
print(f"\n總參數量: {total_params:,}")
print(f"可訓練參數量: {trainable_params:,}")

## 2. VGG (2014)

### 背景

VGG由牛津大學視覺幾何組(Visual Geometry Group)提出，通過使用小卷積核構建更深的網路。

### 核心思想

1. **小卷積核**：全部使用3×3卷積核
2. **更深的網路**：VGG-16有16層，VGG-19有19層
3. **規律的結構**：重複使用相同的基本模塊
4. **逐步降採樣**：使用池化層逐步減小特徵圖尺寸

### 為什麼用3×3卷積？

- 兩個3×3卷積的感受野相當於一個5×5卷積
- 三個3×3卷積的感受野相當於一個7×7卷積
- 但參數更少，非線性更強

**示例計算：**
- 一個7×7卷積：7×7×C×C = 49C² 參數
- 三個3×3卷積：3×(3×3×C×C) = 27C² 參數
- 節省約45%的參數！

### VGG-16結構

```
輸入: 224×224×3

Block 1:
├─ Conv3-64 × 2
└─ MaxPool

Block 2:
├─ Conv3-128 × 2
└─ MaxPool

Block 3:
├─ Conv3-256 × 3
└─ MaxPool

Block 4:
├─ Conv3-512 × 3
└─ MaxPool

Block 5:
├─ Conv3-512 × 3
└─ MaxPool

Classifier:
├─ FC-4096
├─ FC-4096
└─ FC-1000
```

In [ ]:
class VGG(nn.Module):
    """VGG網路實現"""
    
    def __init__(self, vgg_type='VGG11', num_classes=10, in_channels=1):
        super(VGG, self).__init__()
        
        # VGG配置：每個數字表示該塊中卷積層的輸出通道數，'M'表示最大池化
        self.configs = {
            'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
            'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
            'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
            'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
        }
        
        self.features = self._make_layers(self.configs[vgg_type], in_channels)
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            
            nn.Linear(4096, num_classes),
        )
    
    def _make_layers(self, config, in_channels):
        """根據配置構建特徵提取層"""
        layers = []
        
        for v in config:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                layers.extend([
                    nn.Conv2d(in_channels, v, kernel_size=3, padding=1),
                    nn.ReLU(inplace=True)
                ])
                in_channels = v
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# 測試不同的VGG變體
for vgg_type in ['VGG11', 'VGG13', 'VGG16', 'VGG19']:
    model = VGG(vgg_type=vgg_type, num_classes=10)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{vgg_type} 參數量: {total_params:,}")

# 創建VGG11模型
vgg11 = VGG('VGG11', num_classes=10)
x = torch.randn(1, 1, 224, 224)
output = vgg11(x)
print(f"\n輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")

## 3. Network in Network (NiN)

### 核心創新：1×1卷積層

NiN引入了兩個重要概念：

1. **1×1卷積層**：增加非線性，減少參數
2. **全局平均池化**：替代全連接層，減少參數

### 1×1卷積的作用

1. **降維/升維**：調整通道數
2. **增加非線性**：在不改變空間維度的情況下增加網路深度
3. **跨通道信息融合**

### NiN塊結構

```
NiN Block:
├─ Conv (normal)
├─ ReLU
├─ Conv 1×1
├─ ReLU
├─ Conv 1×1
└─ ReLU
```

In [ ]:
def nin_block(in_channels, out_channels, kernel_size, strides, padding):
    """NiN塊：一個卷積層後跟兩個1×1卷積層"""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, strides, padding),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True)
    )

class NiN(nn.Module):
    """Network in Network實現"""
    
    def __init__(self, num_classes=10, in_channels=1):
        super(NiN, self).__init__()
        
        self.model = nn.Sequential(
            # NiN塊1
            nin_block(in_channels, 96, kernel_size=11, strides=4, padding=0),
            nn.MaxPool2d(3, stride=2),
            
            # NiN塊2
            nin_block(96, 256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(3, stride=2),
            
            # NiN塊3
            nin_block(256, 384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(3, stride=2),
            
            # NiN塊4：輸出通道數等於類別數
            nin_block(384, num_classes, kernel_size=3, strides=1, padding=1),
            
            # 全局平均池化層
            nn.AdaptiveAvgPool2d((1, 1)),
            
            # 展平
            nn.Flatten()
        )
    
    def forward(self, x):
        return self.model(x)

# 創建NiN模型
nin = NiN(num_classes=10)
x = torch.randn(1, 1, 224, 224)
output = nin(x)

print("NiN架構:")
print(nin)
print(f"\n輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")

total_params = sum(p.numel() for p in nin.parameters())
print(f"\n總參數量: {total_params:,}")

## 4. GoogLeNet/Inception (2014)

### 核心思想

**Inception模塊**：同時使用多種尺寸的卷積核，讓網路自己學習選擇最優的特徵。

### Inception塊結構

```
輸入
  ├─ 1×1 Conv ────────────────────┐
  ├─ 1×1 Conv → 3×3 Conv ─────────┤
  ├─ 1×1 Conv → 5×5 Conv ─────────┤
  └─ 3×3 MaxPool → 1×1 Conv ──────┤
                                  ↓
                            Concatenate
```

### 設計要點

1. **多尺度特徵提取**：1×1、3×3、5×5卷積並行
2. **1×1卷積降維**：減少計算量
3. **輔助分類器**：緩解梯度消失
4. **全局平均池化**：替代全連接層

In [ ]:
class Inception(nn.Module):
    """Inception模塊"""
    
    def __init__(self, in_channels, c1, c2, c3, c4):
        """
        Args:
            in_channels: 輸入通道數
            c1: 路徑1的輸出通道數（1×1卷積）
            c2: 路徑2的輸出通道數列表 [1×1卷積, 3×3卷積]
            c3: 路徑3的輸出通道數列表 [1×1卷積, 5×5卷積]
            c4: 路徑4的輸出通道數（1×1卷積）
        """
        super(Inception, self).__init__()
        
        # 路徑1：1×1卷積
        self.p1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=1),
            nn.ReLU(inplace=True)
        )
        
        # 路徑2：1×1卷積 → 3×3卷積
        self.p2 = nn.Sequential(
            nn.Conv2d(in_channels, c2[0], kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # 路徑3：1×1卷積 → 5×5卷積
        self.p3 = nn.Sequential(
            nn.Conv2d(in_channels, c3[0], kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )
        
        # 路徑4：3×3最大池化 → 1×1卷積
        self.p4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, c4, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # 在通道維度上連接4個路徑的輸出
        p1 = self.p1(x)
        p2 = self.p2(x)
        p3 = self.p3(x)
        p4 = self.p4(x)
        return torch.cat((p1, p2, p3, p4), dim=1)

# 測試Inception模塊
inception = Inception(in_channels=192, c1=64, c2=[96, 128], c3=[16, 32], c4=32)
x = torch.randn(1, 192, 28, 28)
output = inception(x)
print(f"Inception輸入: {x.shape}")
print(f"Inception輸出: {output.shape}")
print(f"輸出通道數: 64 + 128 + 32 + 32 = {output.shape[1]}")

In [ ]:
class GoogLeNet(nn.Module):
    """GoogLeNet簡化版實現"""
    
    def __init__(self, num_classes=10, in_channels=1):
        super(GoogLeNet, self).__init__()
        
        # 初始卷積層
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        # Inception模塊
        self.inception3a = Inception(192, 64, [96, 128], [16, 32], 32)
        self.inception3b = Inception(256, 128, [128, 192], [32, 96], 64)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.inception4a = Inception(480, 192, [96, 208], [16, 48], 64)
        self.inception4b = Inception(512, 160, [112, 224], [24, 64], 64)
        self.inception4c = Inception(512, 128, [128, 256], [24, 64], 64)
        self.inception4d = Inception(512, 112, [144, 288], [32, 64], 64)
        self.inception4e = Inception(528, 256, [160, 320], [32, 128], 128)
        self.maxpool4 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.inception5a = Inception(832, 256, [160, 320], [32, 128], 128)
        self.inception5b = Inception(832, 384, [192, 384], [48, 128], 128)
        
        # 全局平均池化和分類器
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)
        
        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        x = self.maxpool4(x)
        
        x = self.inception5a(x)
        x = self.inception5b(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# 創建GoogLeNet模型
googlenet = GoogLeNet(num_classes=10)
x = torch.randn(1, 1, 224, 224)
output = googlenet(x)

print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")

total_params = sum(p.numel() for p in googlenet.parameters())
print(f"\n總參數量: {total_params:,}")

## 5. ResNet (2015)

### 問題：深度網路的退化

理論上，更深的網路應該至少和淺網路一樣好（可以學習恆等映射）。但實踐中：
- 訓練誤差反而增加
- 不是過擬合問題
- 是優化困難問題

### 解決方案：残差連接(Residual Connection)

與其學習 H(x)，不如學習 F(x) = H(x) - x

```
      x ─────────────────┐
      │                  │
      ↓                  │
   Conv-ReLU             │ (跳躍連接)
      ↓                  │
   Conv                  │
      ↓                  │
      +──────────────────┘
      ↓
     ReLU
```

### 優勢

1. **易於優化**：恆等映射更容易學習
2. **緩解梯度消失**：梯度可以直接回傳
3. **允許更深的網路**：ResNet-152、ResNet-1000
4. **更好的性能**：ILSVRC 2015冠軍

### ResNet變體

- **ResNet-18/34**：使用基本殘差塊
- **ResNet-50/101/152**：使用瓶頸殘差塊（1×1→3×3→1×1）

In [ ]:
class ResidualBlock(nn.Module):
    """基本殘差塊（用於ResNet-18/34）"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        
        # 主路徑
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                              kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                              kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 跳躍連接（如果維度不匹配，需要調整）
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # 主路徑
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 添加跳躍連接
        out += self.shortcut(x)
        out = self.relu(out)
        
        return out

class BottleneckBlock(nn.Module):
    """瓶頸殘差塊（用於ResNet-50/101/152）"""
    
    expansion = 4  # 輸出通道是中間通道的4倍
    
    def __init__(self, in_channels, mid_channels, stride=1):
        super(BottleneckBlock, self).__init__()
        
        # 1×1卷積降維
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        
        # 3×3卷積
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, 
                              kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        
        # 1×1卷積升維
        self.conv3 = nn.Conv2d(mid_channels, mid_channels * self.expansion,
                              kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels * self.expansion)
        
        self.relu = nn.ReLU(inplace=True)
        
        # 跳躍連接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != mid_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, mid_channels * self.expansion,
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(mid_channels * self.expansion)
            )
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        
        out += self.shortcut(x)
        out = self.relu(out)
        
        return out

# 測試殘差塊
print("=== 基本殘差塊測試 ===")
basic_block = ResidualBlock(64, 64)
x = torch.randn(1, 64, 56, 56)
out = basic_block(x)
print(f"輸入: {x.shape} → 輸出: {out.shape}")

print("\n=== 瓶頸殘差塊測試 ===")
bottleneck = BottleneckBlock(256, 64)
x = torch.randn(1, 256, 56, 56)
out = bottleneck(x)
print(f"輸入: {x.shape} → 輸出: {out.shape}")

In [ ]:
class ResNet(nn.Module):
    """ResNet實現"""
    
    def __init__(self, block, num_blocks, num_classes=10, in_channels=1):
        """
        Args:
            block: 殘差塊類型（ResidualBlock或BottleneckBlock）
            num_blocks: 每個階段的殘差塊數量列表
            num_classes: 分類數量
            in_channels: 輸入通道數
        """
        super(ResNet, self).__init__()
        self.in_channels = 64
        
        # 初始卷積層
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, 
                              stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # 殘差層
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        # 全局平均池化和分類器
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion if hasattr(block, 'expansion') else 512, 
                           num_classes)
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        """構建殘差層"""
        layers = []
        
        # 第一個塊可能需要降採樣
        if hasattr(block, 'expansion'):
            # BottleneckBlock
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
            for _ in range(1, num_blocks):
                layers.append(block(self.in_channels, out_channels))
        else:
            # ResidualBlock
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
            for _ in range(1, num_blocks):
                layers.append(block(out_channels, out_channels))
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x

# 定義不同的ResNet變體
def ResNet18(num_classes=10, in_channels=1):
    return ResNet(ResidualBlock, [2, 2, 2, 2], num_classes, in_channels)

def ResNet34(num_classes=10, in_channels=1):
    return ResNet(ResidualBlock, [3, 4, 6, 3], num_classes, in_channels)

def ResNet50(num_classes=10, in_channels=1):
    return ResNet(BottleneckBlock, [3, 4, 6, 3], num_classes, in_channels)

def ResNet101(num_classes=10, in_channels=1):
    return ResNet(BottleneckBlock, [3, 4, 23, 3], num_classes, in_channels)

# 測試不同的ResNet變體
models = {
    'ResNet-18': ResNet18(),
    'ResNet-34': ResNet34(),
    'ResNet-50': ResNet50(),
}

x = torch.randn(1, 1, 224, 224)
for name, model in models.items():
    output = model(x)
    params = sum(p.numel() for p in model.parameters())
    print(f"{name}: 參數量 = {params:,}, 輸出形狀 = {output.shape}")

## 6. 架構對比與總結

### 參數量和計算複雜度對比

| 模型 | 參數量 | Top-1錯誤率 | 主要特點 |
|------|--------|------------|----------|
| LeNet | ~60K | - | 最早的CNN |
| AlexNet | ~60M | 36.7% | 開啟深度學習時代 |
| VGG-16 | ~138M | 28.5% | 小卷積核，更深 |
| GoogLeNet | ~6.8M | 25.2% | Inception，參數少 |
| ResNet-50 | ~25M | 24.6% | 殘差連接，更深更好 |
| ResNet-152 | ~60M | 23.0% | 最深的網路之一 |

### 設計趨勢

1. **更深的網路** AlexNet(8層) → VGG(16-19層) → ResNet(50-152層)

2. **更小的卷積核** 11×11, 7×7 → 5×5 → 3×3 → 1×1

3. **更少的參數** VGG(138M) → GoogLeNet(6.8M) → ResNet-50(25M)

4. **更好的連接** 串聯 → 並聯(Inception) → 跳躍連接(ResNet)

5. **減少全連接層** 全連接 → 全局平均池化

### 選擇建議

- **準確度優先**：ResNet-101/152
- **速度優先**：MobileNet, EfficientNet
- **記憶體受限**：SqueezeNet, MobileNet
- **理解學習**：LeNet → AlexNet → VGG → ResNet
- **遷移學習**：ResNet-50, EfficientNet

## 練習題

### 1. AlexNet改進
將AlexNet中的Sigmoid激活函數改為ReLU，並使用Dropout，比較性能差異。

### 2. VGG簡化
實現一個簡化版的VGG-11，在CIFAR-10數據集上訓練並評估。

### 3. Inception模塊分析
計算Inception模塊中使用和不使用1×1卷積降維的參數量差異。

### 4. ResNet深度實驗
比較ResNet-18和ResNet-34在相同數據集上的性能，分析殘差連接的作用。

### 5. 混合架構設計
設計一個結合Inception模塊和殘差連接的新架構。

### 6. 計算複雜度分析
計算AlexNet、VGG-16和ResNet-50的FLOPs(浮點運算次數)，理解計算效率。

## 下一步學習

1. **更現代的架構**：MobileNet, EfficientNet, Vision Transformer
2. **數據增強技術**：Cutout, Mixup, AutoAugment
3. **正則化技術**：Dropout, Batch Normalization, Layer Normalization
4. **遷移學習**：使用預訓練模型進行微調
5. **模型可視化**：特徵圖、Grad-CAM、注意力機制
6. **實戰項目**：圖像分類、目標檢測、圖像分割